In [1]:
# from langchain_core.documents import Document
# from langchain_community.document_loaders import PyMuPDFLoader

In [2]:
# document = Document(
#     page_content="",
#     metadata={
#         "source": "",
#         "pages": 18,
#     }
# )
# document

In [3]:
# loader = PyMuPDFLoader(
#     file_path='../data/(Minor Project) PatchFool.pdf',
#     mode='single', #'single', 'page',
# )
# document = loader.load()
# print(document)

# Data Ingestion and Vector DB

In [4]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

c:\Users\Monson\Documents\RAG-Backend-new\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def process_all_PDFs(dir_path):
    docs = []
    pdf_dir = Path(dir_path)
    pdf_files = list(pdf_dir.glob('**/*.pdf'))

    print(f'Found {len(pdf_files)} PDF files')

    for pdf_file in pdf_files:
        print(f"Processing -> {pdf_file.name} ...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            document = loader.load()
            for page in document:
                page.metadata['source_file'] = pdf_file.name
                page.metadata['file_type'] = 'pdf'

            docs.extend(document)
            print(f'Loaded {document} pages')       
        except Exception as e:
            print(f'Error {e}')
    print(f'Total documents loaded: {len(docs)}')
    return docs

In [6]:
folder_path='../data/'
all_pdfs = process_all_PDFs(folder_path)

Found 14 PDF files
Processing -> (AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf ...
Loaded [Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'Certified by IEEE PDFeXpress at August 19, 2023 06:32:47', 'creationdate': '2023-08-17T17:39:02+00:00', 'source': '..\\data\\(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf', 'file_path': '..\\data\\(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf', 'total_pages': 6, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-19T06:32:47+00:00', 'trapped': '', 'modDate': 'D:20230819063247Z', 'creationDate': 'D:20230817173902Z', 'page': 0, 'source_file': '(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf', 'fi

In [31]:
all_pdfs

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'Certified by IEEE PDFeXpress at August 19, 2023 06:32:47', 'creationdate': '2023-08-17T17:39:02+00:00', 'source': '..\\data\\(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf', 'file_path': '..\\data\\(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf', 'total_pages': 6, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-19T06:32:47+00:00', 'trapped': '', 'modDate': 'D:20230819063247Z', 'creationDate': 'D:20230817173902Z', 'page': 0, 'source_file': '(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf', 'file_type': 'pdf'}, page_content='ExposureDiffusion: Learning to Expose for Low-light Image Enhancement\nSupplementary Material\nYufei Wang1, Yi Yu1, Wenhan Yang2, Lanq

# Chunking 

In [8]:
def split_docs(docs, chunk_size=500, chunk_overlap=200):
    enriched_docs = []
    for doc in docs:
        meta = doc.metadata
        content = doc.page_content

        # Try to enrich with author/title if available
        title = meta.get("title", "")
        authors = meta.get("authors", "")
        if title or authors:
            enriched_text = f"Title: {title}\nAuthors: {authors}\n\n{content}"
        else:
            enriched_text = content

        doc.page_content = enriched_text
        enriched_docs.append(doc)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', ' ', '\n', '']
    )

    all_chunks = text_splitter.split_documents(enriched_docs)
    print(f"Formed {len(all_chunks)} chunks from {len(docs)} docs")
    return all_chunks

In [9]:
split_documents = split_docs(all_pdfs)

Formed 2391 chunks from 213 docs


In [29]:
split_documents[0].metadata

{'producer': 'pdfTeX-1.40.24',
 'creator': 'Certified by IEEE PDFeXpress at August 19, 2023 06:32:47',
 'creationdate': '2023-08-17T17:39:02+00:00',
 'source': '..\\data\\(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf',
 'file_path': '..\\data\\(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf',
 'total_pages': 6,
 'format': 'PDF 1.5',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2023-08-19T06:32:47+00:00',
 'trapped': '',
 'modDate': 'D:20230819063247Z',
 'creationDate': 'D:20230817173902Z',
 'page': 0,
 'source_file': '(AML Project) Wang_ExposureDiffusion_Learning_to_Expose_for_Low-light_Image_Enhancement_ICCV_2023_supplementary_paper.pdf',
 'file_type': 'pdf'}

# Embeddings and Vector DB

In [11]:
import os
import uuid
import chromadb
import numpy as np

from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Tuple

In [12]:
class EmbeddingManager:
    # model_name is the Hugggingface one
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        # all-MiniLM-L6-v2 : 384 dim
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f'Loading model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f'Loaded model: {self.model.get_sentence_embedding_dimension()} dimensional')
        except Exception as e:
            print(f'Error loading model {self.model_name}, {e}')
            raise

    def generate_embeddings(self, text: List[str]):
        if not self.model:
            raise ValueError('Model not loaded !!!')
        print(f'Generating embeddings for {self.model_name}')
        embeddings = self.model.encode(text)
        print(f'Generated (with shape) {embeddings.shape} dimensional embeddings')
        return embeddings.tolist()

    def get_sentence_embedding_dimension(self):
        if not self.model:
            raise ValueError('Model not loaded !!!')
        return self.model.get_sentence_embedding_dimension()

In [13]:
embedding_manager = EmbeddingManager()

texts = [doc.page_content for doc in split_documents] 

# Normalizing distances
embeddings = embedding_manager.generate_embeddings(texts)
# embeddings = np.array(embeddings)
# embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

Loading model: all-MiniLM-L6-v2
Loaded model: 384 dimensional
Generating embeddings for all-MiniLM-L6-v2
Generated (with shape) (2391, 384) dimensional embeddings


# Vector store

In [14]:
class VectorStore:
    def __init__(self, collection_name: str, persist_dir: str='../data/vector_store'):
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_dir, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_dir)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'hnsw:space': 'cosine'}
            )
            print(f'VectorStore for {self.collection_name} initialized successfully')
            print(f'Existing numner of docs in the colelction: {self.collection.count()}')
        
        except Exception as e:
            print(f'Error in initializing Vector Store {e}')
            raise 

    def add_docs(self, docs: List[Any], embeddings: np.ndarray):
        if len(docs) != len(embeddings):
            raise ValueError(f'Number of docs ({len(docs)}) must match number of embeddings ({len(embeddings)})...')
        
        print(f'Adding {len(docs)} docs to vectorStore')
    
        ids = []
        metadatas = []
        doc_list = []
        embedding_list = []
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            id = f'doc_{uuid.uuid4().hex[:8]}_{i}'
            ids.append(id)

            doc_list.append(doc.page_content)
            
            embedding_list.append(embedding)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

        try:
            print(f'Adding docs and embeddings to collection {self.collection_name}')
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=doc_list
            )
            print(f'Successfuly added {len(docs)} documents and embeddings to the collection.')
            print(f'Total docs and embeddings in {self.collection_name} is {self.collection.count()}')
        
        except Exception as e:
            raise ValueError(f'Could not add docs and embeds to collection {self.collection_name}')
        

In [15]:
vectorStore = VectorStore(collection_name='PatchFool_Paper', persist_dir='../data/vector_store/PatchFool')
# vectorStore.add_docs(docs=split_documents, embeddings=embeddings)

VectorStore for PatchFool_Paper initialized successfully
Existing numner of docs in the colelction: 204


# Retriever

In [16]:
class Retriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        # score_threshold: Minimum similarity score threshold
        
        print(f'Retrieving docs for query: {query}')
        print(f'Top k = {k} and score threshold {score_threshold}')

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        # query_embedding = np.array(query_embedding)
        # query_embedding = query_embedding / np.linalg.norm(query_embedding, keepdims=True)
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=k
            )

            retrieved_docs = []

            if results['documents'] and results['metadatas'] and results['distances'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    # print(i, similarity_score, distance)
                    if similarity_score > score_threshold:
                        retrieved_docs.append({
                            'id': id,
                            'document': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                print(f'Retrieved {len(retrieved_docs)} after filtering')
            else:
                print(f'No docs found --^^--')
            return retrieved_docs
        except Exception as e:
            raise ValueError(f'Could not retrieve documents : {e}')

In [17]:
retriever = Retriever(vector_store=vectorStore, embedding_manager=embedding_manager)

question = "How better is patchfool compared to its predecessors, give metrics as well?"
retrieved_docs = retriever.retrieve(question)

Retrieving docs for query: How better is patchfool compared to its predecessors, give metrics as well?
Top k = 5 and score threshold 0.0
Generating embeddings for all-MiniLM-L6-v2
Generated (with shape) (1, 384) dimensional embeddings
Retrieved 5 after filtering


In [18]:
retrieved_docs[0]

{'id': 'doc_c71078ff_104',
 'document': 'to the perturbation density, we evaluate our proposed Mild Patch-Fool in Sec. 4.6.\n4.6\nBENCHMARK AGAINST MILD PATCH-FOOL\nSetup. To study the influence of the perturbation strength within each patch, we evaluate our\nproposed Mild Patch-Fool in Sec. 4.6 with L2 or L∞constraints on the patch-wise perturbations with\n8',
 'metadata': {'file_path': '..\\data\\(Minor Project) PatchFool.pdf',
  'page': 7,
  'context_length': 315,
  'source': '..\\data\\(Minor Project) PatchFool.pdf',
  'format': 'PDF 1.5',
  'author': '',
  'subject': '',
  'trapped': '',
  'title': '',
  'file_type': 'pdf',
  'keywords': '',
  'source_file': '(Minor Project) PatchFool.pdf',
  'modDate': 'D:20250107011300Z',
  'creationDate': 'D:20250107011300Z',
  'doc_index': 104,
  'creator': 'LaTeX with hyperref',
  'moddate': '2025-01-07T01:13:00+00:00',
  'producer': 'pdfTeX-1.40.25',
  'creationdate': '2025-01-07T01:13:00+00:00',
  'total_pages': 18},
 'similarity_score': 0.

# LLM Generation

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

True

In [20]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [21]:
# llm.invoke('Write me a song')

def simple_RAG(query: str, retriever: Retriever, llm: ChatGoogleGenerativeAI, top_k: int = 5):
    results = retriever.retrieve(query=query)
    context = "\n\n".join([i['document'] for i in results])
    if not context:
        return 'No relevant context found.'
    
    template = """
            You are a helpful assistant, for reasearch papers. You haev research papers as data, so you need to be aware of how a research paper is structured, (like where to find the authors of the paper and stuff etc.). Use the following pieces of context to answer the question at the end.
            If you don't know the answer, just say that you don't know, don't try to make up an answer.
            Use five sentences maximum and keep the answer as concise as possible.

            Context: {context}

            Question: {query}

            Helpful Answer:
        """
    
    response = llm.invoke([template.format(context=context, query=query)])
    return response.content

In [22]:
answer = simple_RAG(query=question, retriever=retriever, llm=llm, top_k=5)

Retrieving docs for query: How better is patchfool compared to its predecessors, give metrics as well?
Top k = 5 and score threshold 0.0
Generating embeddings for all-MiniLM-L6-v2
Generated (with shape) (1, 384) dimensional embeddings
Retrieved 5 after filtering


In [23]:
answer

'Based on the provided context, the document does not explicitly introduce "predecessors" to Patch-Fool and provide a direct comparison with metrics demonstrating how much better Patch-Fool performs. The text discusses variants like "Mild Patch-Fool" and "Sparse Patch-Fool," which are either benchmarks or variations of Patch-Fool itself, rather than predecessors. It also mentions that "attention-aware patch selection is the most effective strategy in most cases" among three strategies for attacking ViTs, but it does not name the other strategies or provide comparative metrics.'